This is the Three Stage Model with Bert (0.733 Macro F1 on leaderbord)

In [ ]:
# Import Libraries and Frameworks

import pandas as pd
import numpy as np
import re

from collections import Counter, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer

In [ ]:
# Paths

DEV_IN_PATH  = "../../data/processed/development_processed.csv"
EVAL_IN_PATH = "../../data/processed/evaluation_processed.csv"
SUB_OUT      = "../../data/submission/submission_bert_three_stage.csv"

In [ ]:
# Best Parameters

MIN_RULE_SUPPORT = 34
MIN_RULE_PURITY  = 0.926328564964768

WORD_NG_MAX = 2
CHAR_NG_MAX = 5
MIN_DF      = 2
MAX_DF      = 0.8782583211530898
C_VALUE     = 0.64491922705094

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

In [ ]:
# Load Data 

df_dev  = pd.read_csv(DEV_IN_PATH)
df_eval = pd.read_csv(EVAL_IN_PATH)

FEATURES_BASE = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES_BASE]
y_dev  = df_dev["label"].astype(int)
X_eval = df_eval[FEATURES_BASE]

In [ ]:
# Stage 3 - Bert Embeddings (on processed Text)

print("Encoding Sentence-BERT embeddings...")

bert_model = SentenceTransformer("all-MiniLM-L6-v2")

def bert_input(df):
	# text è già title + article
	return df["text"].str.slice(0, 2000).tolist()

bert_dev = bert_model.encode(
	bert_input(df_dev),
	batch_size=32,
	show_progress_bar=True,
	normalize_embeddings=True
)

bert_eval = bert_model.encode(
	bert_input(df_eval),
	batch_size=32,
	show_progress_bar=True,
	normalize_embeddings=True
)

BERT_DIM = bert_dev.shape[1]
bert_cols = [f"bert_{i}" for i in range(BERT_DIM)]

df_dev[bert_cols]  = bert_dev
df_eval[bert_cols] = bert_eval


# STAGE 1 — RULE MINING (ONLY ON ARTICLE)

def tokenize_for_rules(text):
	# deterministico per HTML / URL / boilerplate
	return re.findall(r"[a-z0-9_:/\.]+", text.lower())


def mine_pure_rules(texts, labels):
	counts = defaultdict(lambda: Counter())

	for txt, y in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][int(y)] += 1

	rule_token_to_class = {}
	rule_meta = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < MIN_RULE_SUPPORT:
			continue

		best_class, best_freq = c.most_common(1)[0]
		purity = best_freq / total

		if purity >= MIN_RULE_PURITY:
			rule_token_to_class[tok] = int(best_class)
			rule_meta[tok] = (purity, total)

	return rule_token_to_class, rule_meta


def apply_rules(texts, rule_token_to_class, rule_meta):
	rule_pred = np.full(len(texts), -1, dtype=int)

	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rule_token_to_class]
		if not hits:
			continue

		hits.sort(
			key=lambda t: (rule_meta[t][0], rule_meta[t][1]),
			reverse=True
		)

		rule_pred[i] = rule_token_to_class[hits[0]]

	return rule_pred

In [ ]:
#Stage 2 + 3 Model (TF-IDF + Num + Bert )

def make_model():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
			("bert", "passthrough", bert_cols),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=3000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf),
	])


In [ ]:
# Train 
model = make_model()
model.fit(df_dev[FEATURES_BASE + bert_cols], y_dev)

rule_token_to_class, rule_meta = mine_pure_rules(
	df_dev["article"],
	y_dev
)

print("Rules mined:", len(rule_token_to_class))


In [ ]:
# Predict 

model_pred = model.predict(df_eval[FEATURES_BASE + bert_cols])

rule_pred = apply_rules(
	df_eval["article"],
	rule_token_to_class,
	rule_meta
)

final_pred = model_pred.copy()
mask = rule_pred != -1
final_pred[mask] = rule_pred[mask]

print(f"Rule coverage on eval: {mask.mean():.4f}")

In [ ]:
# Submission 

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": final_pred.astype(int)
})

submission.to_csv(SUB_OUT, index=False)
print("Saved submission to:", SUB_OUT)